# 08 — Balayage du lissage de `te_origin`

`te_origin` (taux de fraude de l'émetteur) pèse 33% de l'importance. Il est peut-être
SUR-LISSÉ (smoothing=30) -> on écrase le signal des comptes peu actifs.
On teste `smoothing ∈ {3, 10, 30}` sur CatBoost (meilleur modèle, cf. 07) et on garde le meilleur.

Référence : CatBoost smoothing=30 -> last 0.3607, LB~ 0.3437. Cible : > 0.3528 (l'ami).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, seed_everything, make_submission
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
op03 = op03_mask(train).to_numpy()
y_all = train[C.TARGET].to_numpy()
folds_full = list(time_folds(train[C.PERIOD]))

In [ ]:
EPS = 1e-6
WINDOWS = (5, 10, 20)

def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]
    f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)

def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X

def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    X = add_freq(X, df.reset_index(drop=True), ref)
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    rt = recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)
    return pd.concat([X, beh, rec, rt], axis=1)

def feats_train(df, ref, smoothing):
    X = base_build(df, ref)
    X["te_origin"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=smoothing)
    return X

def feats_apply(df, ref, smoothing):
    X = base_build(df, ref)
    mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=smoothing)
    X["te_origin"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm)
    return X

def make_cat():
    from catboost import CatBoostClassifier
    return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                              learning_rate=0.05, iterations=600, random_seed=42, verbose=False)

## Balayage du lissage

In [ ]:
def run_cv(smoothing):
    oof = np.zeros(len(train)); per_fold = []
    for tr_idx, va_idx in folds_full:
        tr_op = tr_idx[op03[tr_idx]]; va_op = va_idx[op03[va_idx]]
        ref = train.iloc[tr_op]
        Xtr = feats_train(train.iloc[tr_op], ref, smoothing)
        Xva = feats_apply(train.iloc[va_op], ref, smoothing)
        m = make_cat().fit(Xtr, y_all[tr_op])
        oof[va_op] = m.predict_proba(Xva)[:, 1]
        per_fold.append(evaluate_ap(y_all[va_op], oof[va_op]))
    return per_fold, oof

results = {}
for s in [3, 10, 30]:
    pf, oof = run_cv(s)
    results[s] = (pf, oof)
    print(f"smoothing={s:2d} : global {np.mean(pf):.4f} | recent(2) {np.mean(pf[-2:]):.4f} | last {pf[-1]:.4f} | LB~ {pf[-1]-0.017:.4f}")

best_s = max(results, key=lambda s: np.mean(results[s][0][-2:]))
print("\nMeilleur lissage (sur recent2) :", best_s)

## Soumission au meilleur lissage (sans calibration)

In [ ]:
S = best_s
ref_full = train.iloc[np.where(op03)[0]]
Xf = feats_train(ref_full, ref_full, S); yf = y_all[op03]
final = make_cat().fit(Xf, yf)
te_op = op03_mask(test).to_numpy()
test_op = test.iloc[np.where(te_op)[0]]
Xte = feats_apply(test_op, ref_full, S)
proba = final.predict_proba(Xte)[:, 1]
full = np.zeros(len(test)); full[te_op] = proba
path = make_submission(test[C.ID], full, f"08_te_smooth{S}")
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print("soumission écrite :", path, "| proba>0 :", int((sub['target'] > 0).sum()))